# Potter Airlines Dynamic Revenue Management System
### Team 6 Project
Emma Oh, Kunbo Ma, Rosie Wang, Vaibhav Verma, Yuanyuan Zhong

# Step 1: Connecting to SQLite Database with Flights Dataset

In [83]:
import sqlite3
import json
import pandas as pd
from datetime import datetime
import math
from numbers import Integral, Real

DB_NAME = "potter_airlines.db"
JSON_FILE = "data/flights.json"

### Create the Flights Table

In [84]:
def create_table():
    """Create the flights table if it does not already exist."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("DROP TABLE IF EXISTS flights")
        
        conn.execute("""
            CREATE TABLE flights (
                flight_id TEXT PRIMARY KEY,
                origin TEXT NOT NULL,
                destination TEXT NOT NULL,
                departure_date TEXT NOT NULL,
                days_until_departure INTEGER NOT NULL
                    CHECK (days_until_departure >= 0),
                base_fare REAL NOT NULL
                    CHECK (base_fare > 0),
                seats_remaining INTEGER NOT NULL
                    CHECK (seats_remaining >= 0 AND seats_remaining <= capacity),
                capacity INTEGER NOT NULL
                    CHECK (capacity > 0),
                route_popularity REAL NOT NULL
                    CHECK (route_popularity >= 0 AND route_popularity <= 1),
                international INTEGER NOT NULL
                    CHECK (international IN (0, 1))
            )
        """)

### Load Flight Data from JSON

In [85]:
def load_json(filename=JSON_FILE):
    """Load flight data from the JSON file."""

    with open(filename, "r") as file:
        return json.load(file)

### Insert All Flights from JSON

In [86]:
def insert_all_flights(flights):
    """Insert all flights from the JSON dataset."""

    with sqlite3.connect(DB_NAME) as conn:
        for flight in flights:
            conn.execute("""
                INSERT OR IGNORE INTO flights
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                flight["flight_id"],
                flight["origin"],
                flight["destination"],
                flight["departure_date"],
                flight["days_until_departure"],
                flight["base_fare"],
                flight["seats_remaining"],
                flight["capacity"],
                flight["route_popularity"],
                flight["international"]
            ))

### SELECT - All Flights

In [87]:
def get_all_flights():
    """Retrieve all flights from the database."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("SELECT * FROM flights")
        return cursor.fetchall()

### Check Loaded Data

In [88]:
create_table()

loaded_flights = load_json()
insert_all_flights(loaded_flights)

print(f"{len(loaded_flights)} flights loaded from JSON.")
print(f"{len(get_all_flights())} flights stored in SQLite.")

# Check that all flights were successfully stored
assert len(get_all_flights()) == len(loaded_flights)

print("Database setup completed successfully.")

1044 flights loaded from JSON.
1044 flights stored in SQLite.
Database setup completed successfully.


In [89]:
flight_data = pd.DataFrame(loaded_flights)
flight_data.head()

,flight_id,origin,destination,departure_date,days_until_departure,base_fare,seats_remaining,capacity,route_popularity,international
0,PA1000,Vancouver,Calgary,02-28-2027,119,163,57,91,0.87,0
1,PA1001,Vancouver,Calgary,11-22-2026,21,163,24,109,0.87,0
2,PA1002,Vancouver,Edmonton,12-24-2026,53,175,64,96,0.65,0
3,PA1003,Vancouver,Edmonton,03-04-2027,123,175,1,111,0.65,0
4,PA1004,Vancouver,Saskatoon,12-30-2026,59,298,35,86,0.68,0


# Step 2: Introduce Flight Class and Pricing Logic
## Pricing method

The model starts with the route's `base_fare` and adjusts it using five transparent business factors. The factor values are project assumptions designed to make fares respond predictably to booking time, demand, aircraft occupancy, travel season, and cabin class. **This description part is AI-generated**

| Factor | Input and pricing rule |
|---|---|
| Time | The multiplier rises from 0.90 for bookings more than 90 days before departure to 1.35 for bookings made 0-3 days before departure. |
| Demand | `route_popularity` is divided into bands from 0.00-1.00. The multiplier ranges from 0.85 for very low popularity to 1.20 for very high popularity. |
| Capacity | Occupancy is calculated as `(capacity - seats_remaining) / capacity`. The multiplier ranges from 0.85 when less than 25% of seats are occupied to 1.45 when at least 95% are occupied. |
| Seasonal | Christmas and New Year use 1.20, summer uses 1.15, March uses 1.10, the January-February low season uses 0.95, and other dates use 1.00. |
| Class | Economy is the default at 1.00. Business class uses 1.75. |

### Why these coefficients were chosen

These coefficients are rule-based assumptions rather than estimates from historical sales. The dataset is synthetic and contains flight characteristics, but it does not contain observed customer purchases or final market prices that could be used to statistically estimate each factor. The selected multipliers therefore create clear and moderate price movements that can be explained from the available fields.

- **Time:** `days_until_departure` ranges from 0 to 151 days. The dataset contains 384 flights more than 90 days from departure, but only 29 flights in the 0-3 day range and 25 in the 4-7 day range. The wider early-booking bands cover the more common observations and provide modest discounts, while the smaller last-minute bands apply progressively larger premiums as departure approaches.
- **Demand:** Almost all `route_popularity` values fall between 0.50 and 0.89, distributed across four similar-sized 0.10 bands. Using 1.00 for the 0.60-0.69 band provides a neutral reference point, with gradual discounts below it and premiums above it.
- **Capacity:** Aircraft occupancy is spread across the full 0%-100% range. Lower-occupancy flights receive discounts to encourage bookings, while the premium increases more sharply above 85% occupancy because available seats have become scarce.
- **Seasonal:** The dataset covers November 2026 through April 2027, including 162 Christmas/New Year flights, 345 January-February low-season flights, 203 March flights, and 334 regular-date flights. These groups support a holiday premium, a post-holiday discount, and a smaller March-break premium. The summer multiplier is retained as a general rule for future data, although the current dataset contains no summer departures.
- **Class:** Economy is the 1.00 baseline, while the 1.75 Business multiplier is a simple project assumption that creates a clear service-level premium without introducing additional unsupported categories.

### Validation rules

- `calculate_time_factor()` requires a whole-number `days_until_departure` value that is zero or greater.
- `calculate_demand_factor()` requires a finite numeric `route_popularity` value between 0 and 1.
- `calculate_capacity_factor()` requires positive whole-number capacity and remaining seats between zero and capacity.
- `calculate_seasonal_factor()` requires a real calendar date written exactly in `MM-DD-YYYY` format.
- `calculate_class_factor()` accepts only the text values `economy` and `business`, ignoring capitalization and surrounding spaces.
- `validate_flight_data()` checks all required JSON fields and calls the relevant factor functions before a Flight object or price is created.
- `calculate_final_price()` requires a positive base fare and asserts that the bounded result stays within its permitted minimum and maximum.

After all factors are multiplied, the result is bounded between `base_fare * 0.60 * class_factor` and `base_fare * 2.50 * class_factor`, then rounded to two decimal places. The `international` field is retained as flight information but is not a separate multiplier because domestic and international differences are already reflected in the generated base fares.

Relative price bounds are used instead of fixed dollar limits because base fares in the dataset range from $160 to $1,697. Scaling the bounds with both `base_fare` and `class_factor` preserves meaningful differences between short domestic routes, long international routes, Economy fares, and Business fares.

## Flight class

The `Flight` class stores one record from `flights.json`. `Flight.from_dict()` converts a JSON dictionary into a Flight object, `to_dict()` returns the original dictionary structure, and `calculate_price()` produces an Economy or Business quote using the shared pricing functions.

The **@classmethod** converts each flight dictionary loaded from the JSON dataset directly into a Flight object, avoiding the need to pass every attribute manually.

The examples at the end of the notebook load one domestic flight and one international flight from the dataset. The tested quotes are PA1000 Vancouver-Calgary at $152.26 Economy and $266.45 Business, and PA2000 Vancouver-Los Angeles at $316.01 Economy and $553.01 Business.

In [90]:
class Flight:
    #Represent one Potter Airlines flight and calculate its fare.
    
    CLASS_FACTORS = {
        "economy": 1.00,
        "business": 1.75,
    }
    
    REQUIRED_FLIGHT_FIELDS = (
        "flight_id",
        "origin",
        "destination",
        "departure_date",
        "days_until_departure",
        "base_fare",
        "seats_remaining",
        "capacity",
        "route_popularity",
        "international",
    )

    def __init__(
        self,
        flight_id,
        origin,
        destination,
        departure_date,
        days_until_departure,
        base_fare,
        seats_remaining,
        capacity,
        route_popularity,
        international,
    ):
        self.flight_id = flight_id
        self.origin = origin
        self.destination = destination
        self.departure_date = departure_date
        self.days_until_departure = days_until_departure
        self.base_fare = base_fare
        self.seats_remaining = seats_remaining
        self.capacity = capacity
        self.route_popularity = route_popularity
        self.international = international

    @classmethod 
    def from_dict(cls, flight_data):
        #Create a Flight object from one flight dictionary.
        
         # Validate one complete flight record before creating an object or calculating a price to make sure the flight data is reasonable. 
        if not isinstance(flight_data, dict):
            raise TypeError("flight must be a dictionary.")
        
        missing_fields = [field for field in cls.REQUIRED_FLIGHT_FIELDS if field not in flight_data]
        if missing_fields:
            raise ValueError("flight is missing required fields: " + ", ".join(missing_fields))

        for field_name in ("flight_id", "origin", "destination"):
            value = flight_data[field_name]
            if not isinstance(value, str):
                raise TypeError(f"{field_name} must be text.")
            if not value.strip():
                raise ValueError(f"{field_name} cannot be empty.")
                    
        return cls(
            flight_id=flight_data["flight_id"],
            origin=flight_data["origin"],
            destination=flight_data["destination"],
            departure_date=flight_data["departure_date"],
            days_until_departure=flight_data["days_until_departure"],
            base_fare=flight_data["base_fare"],
            seats_remaining=flight_data["seats_remaining"],
            capacity=flight_data["capacity"],
            route_popularity=flight_data["route_popularity"],
            international=flight_data["international"],
        )
    #easier to convert JSON flight records into consistent Flight objects that can be reused later for pricing, 
    #database operations, filtering, and system integration.

    def to_dict(self):
        """Return the flight information in the original dictionary format."""
        return {
            "flight_id": self.flight_id,
            "origin": self.origin,
            "destination": self.destination,
            "departure_date": self.departure_date,
            "days_until_departure": self.days_until_departure,
            "base_fare": self.base_fare,
            "seats_remaining": self.seats_remaining,
            "capacity": self.capacity,
            "route_popularity": self.route_popularity,
            "international": self.international,
        }
        
    # These two functions will be applied in following factor calculation functions multiple times to do the validation. 
    def _validate_number(self, value, field_name):
        """Check that a numeric field is a real, finite number rather than text, Boolean, NaN, or infinity."""
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError(f"{field_name} must be a number.")
        if not math.isfinite(float(value)):
            raise ValueError(f"{field_name} must be a finite number.")


    def _validate_integer(self, value, field_name):
        """Check fields that represent whole units such as days or seat counts."""
        if isinstance(value, bool) or not isinstance(value, Integral):
            raise TypeError(f"{field_name} must be a whole number.")
        
    def _calculate_time_factor(self):
        """Return a fare multiplier based on how soon the flight departs."""
        self._validate_integer(self.days_until_departure, "days_until_departure")
        if self.days_until_departure < 0:
            raise ValueError("days_until_departure cannot be negative.")

        if self.days_until_departure <= 3:
            return 1.35
        if self.days_until_departure <= 7:
            return 1.25
        if self.days_until_departure <= 14:
            return 1.15
        if self.days_until_departure <= 30:
            return 1.08
        if self.days_until_departure <= 60:
            return 1.00
        if self.days_until_departure <= 90:
            return 0.95
        return 0.90
    
    def _calculate_demand_factor(self):
        """Return a fare multiplier based on route popularity."""
        self._validate_number(self.route_popularity, "route_popularity")
        if not 0 <= self.route_popularity <= 1:
            raise ValueError("route_popularity must be between 0 and 1.")

        if self.route_popularity < 0.50:
            return 0.85
        if self.route_popularity < 0.60:
            return 0.90
        if self.route_popularity < 0.70:
            return 1.00
        if self.route_popularity < 0.80:
            return 1.08
        if self.route_popularity < 0.90:
            return 1.15
        return 1.20
    
    def _calculate_capacity_factor(self):
        """Return a fare multiplier based on the proportion of seats already sold."""
        self._validate_integer(self.seats_remaining, "seats_remaining")
        self._validate_integer(self.capacity, "capacity")
        if self.capacity <= 0:
            raise ValueError("capacity must be greater than zero.")
        if not 0 <= self.seats_remaining <= self.capacity:
            raise ValueError("seats_remaining must be between 0 and capacity.")

        occupancy_rate = (self.capacity - self.seats_remaining) / self.capacity

        if occupancy_rate < 0.25:
            return 0.85
        if occupancy_rate < 0.50:
            return 0.95
        if occupancy_rate < 0.70:
            return 1.05
        if occupancy_rate < 0.85:
            return 1.15
        if occupancy_rate < 0.95:
            return 1.30
        return 1.45
    
    def _calculate_seasonal_factor(self):
        """Return a fare multiplier for holiday, summer, spring-break, or regular travel."""
        if not isinstance(self.departure_date, str):
            raise TypeError("departure_date must be text in MM-DD-YYYY format.")
        try:
            departure = datetime.strptime(self.departure_date, "%m-%d-%Y")
        except ValueError as error:
            raise ValueError(
                "departure_date must be a valid date in MM-DD-YYYY format."
            ) from error
        if departure.strftime("%m-%d-%Y") != self.departure_date:
            raise ValueError(
                "departure_date must use exactly MM-DD-YYYY format."
            )

        month = departure.month
        day = departure.day

        if (month == 12 and day >= 15) or (month == 1 and day <= 5):
            return 1.20
        if month in (6, 7, 8):
            return 1.15
        if month == 3:
            return 1.10
        if month in (1, 2):
            return 0.95
        return 1.00
    
    def _calculate_class_factor(self, fare_class="economy"):
        """Return the multiplier for economy or business class."""
        if not isinstance(fare_class, str):
            raise TypeError("fare_class must be text.")
        normalized_class = fare_class.strip().lower()
        if normalized_class not in self.CLASS_FACTORS:
            raise ValueError("fare_class must be either economy or business.")
        return self.CLASS_FACTORS[normalized_class]
    
    def calculate_final_price(self, fare_class="economy"):
        """Combine all fare factors and return the bounded final price."""
            
        self._validate_number(self.base_fare, "base_fare")
        if self.base_fare <= 0:
            raise ValueError("base_fare must be greater than zero.")
        
        self._validate_integer(self.international, "international")
        if self.international not in (0, 1):
            raise ValueError("international must be either 0 or 1.")

        time_factor = self._calculate_time_factor()
        demand_factor = self._calculate_demand_factor()
        capacity_factor = self._calculate_capacity_factor()
        seasonal_factor = self._calculate_seasonal_factor()
        class_factor = self._calculate_class_factor(fare_class)

        raw_price = (self.base_fare*time_factor*demand_factor*capacity_factor*seasonal_factor*class_factor)

        minimum_fare = self.base_fare * 0.60 * class_factor
        maximum_fare = self.base_fare * 2.50 * class_factor
        final_price = max(minimum_fare, min(raw_price, maximum_fare))
        assert minimum_fare <= final_price <= maximum_fare

        return round(final_price, 2)

## Step 3: Instantiate Flight Class & Calculate Price
#### *Flights leaving from Toronto*

In [91]:
# first load all flights and instantiate Flight 
all_flights = []
for flight_data in loaded_flights:
  flight = Flight.from_dict(flight_data)
  all_flights.append(flight)
  
# filter for flights originating from Toronto
toronto_flights = []
for flight in all_flights:
  if flight.origin == "Toronto":
    toronto_flights.append(flight)

In [92]:
# calculate prices for the Toronto flights 
toronto_prices = []
for toronto_flight in toronto_flights:
  toronto_data = toronto_flight.to_dict()
  toronto_data["economy_price"] = toronto_flight.calculate_final_price()
  toronto_data["business_price"] = toronto_flight.calculate_final_price("business")
  toronto_prices.append(toronto_data)

# create dataframe for data interpretation
toronto_df = pd.DataFrame(toronto_prices)
# sample random 5 domestic and international Toronto flights
domestic_sample = toronto_df[toronto_df["international"]==0].sample(5)
international_sample = toronto_df[toronto_df["international"]==1].sample(5)
sample_toronto = pd.concat([domestic_sample, international_sample])

sample_toronto

,flight_id,origin,destination,departure_date,days_until_departure,base_fare,seats_remaining,capacity,route_popularity,international,economy_price,business_price
2,PA1122,Toronto,Calgary,12-07-2026,36,409,7,158,0.66,0,593.05,1037.84
15,PA1173,Toronto,Montreal,01-30-2027,90,178,38,106,0.88,0,193.98,339.46
1,PA1121,Toronto,Vancouver,01-14-2027,74,521,96,162,0.80,0,513.70,898.97
18,PA1204,Toronto,Halifax,01-02-2027,62,285,9,99,0.83,0,485.73,850.02
21,PA1207,Toronto,St. John's,11-29-2026,28,169,85,105,0.57,0,139.63,244.35
28,PA2092,Toronto,Boston,12-13-2026,42,314,135,189,0.87,1,343.04,600.33
36,PA2676,Toronto,Tokyo,03-14-2027,133,1452,116,293,0.82,1,1735.76,3037.57
29,PA2093,Toronto,Boston,12-20-2026,49,314,45,188,0.87,1,498.32,872.06
34,PA2212,Toronto,Mexico City,12-21-2026,50,783,7,141,0.59,1,1226.18,2145.81
45,PA2749,Toronto,Singapore,12-16-2026,45,1586,238,283,0.67,1,1617.72,2831.01


## Filter & Sort Toronto Domestic Flights

In [93]:
# filter for just domestic flights and sort by economy price (cheapest -> most expensive)
toronto_domestic = toronto_df[toronto_df["international"]==0].sort_values(by="economy_price", ascending=True)
# show top 10 cheapest Toronto domestic flights
toronto_domestic.head(10)

,flight_id,origin,destination,departure_date,days_until_departure,base_fare,seats_remaining,capacity,route_popularity,international,economy_price,business_price
11,PA1169,Toronto,Winnipeg,02-11-2027,102,167,66,101,0.64,0,135.65,237.38
21,PA1207,Toronto,St. John's,11-29-2026,28,169,85,105,0.57,0,139.63,244.35
20,PA1206,Toronto,St. John's,12-27-2026,56,169,51,93,0.57,0,173.39,303.44
12,PA1170,Toronto,Ottawa,03-08-2027,127,180,75,107,0.75,0,182.83,319.96
14,PA1172,Toronto,Montreal,11-25-2026,24,178,77,89,0.88,0,187.91,328.85
15,PA1173,Toronto,Montreal,01-30-2027,90,178,38,106,0.88,0,193.98,339.46
17,PA1175,Toronto,Quebec City,03-24-2027,143,241,53,86,0.53,0,203.99,356.99
13,PA1171,Toronto,Ottawa,11-24-2026,23,180,40,97,0.75,0,220.45,385.79
16,PA1174,Toronto,Quebec City,01-13-2027,73,241,22,90,0.53,0,225.12,393.95
10,PA1168,Toronto,Winnipeg,03-09-2027,128,167,4,119,0.64,0,239.73,419.52


# Step 4: Update Data
*Purchased cheapest flight for Toronto -> Winnipeg (PA1169), so we need to update the data!*

In [94]:
def update_seats(flight_id, seats_purchased):
    """Update the number of seats remaining for a flight."""
    if seats_purchased <= 0: 
        raise ValueError("seats_purchased must be greater than 0.")
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            UPDATE flights
            SET seats_remaining = seats_remaining - ?
            WHERE flight_id = ?
            AND seats_remaining >= ?
            """, (seats_purchased, flight_id, seats_purchased))
        
        if cursor.rowcount == 0:
            raise ValueError("Flight not found or not enough seats are available.")

In [95]:
flight_id_update = input(
    "Please provide the Flight ID that you purchased tickets for: "
).upper()

num_seats_update = int(input(
    "How many tickets (seats) are you purchasing?: "
))

update_seats(flight_id_update, num_seats_update)

In [96]:
def get_flight(flight_id):
    """Retrieve one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            SELECT * FROM flights
            WHERE flight_id = ?
        """, (flight_id,))

        row = cursor.fetchone()

        if row is None:
            print("Flight not found.")
            return None, None
            
        columns = [description[0] for description in cursor.description]
    return row, columns

In [97]:
row, columns = get_flight(flight_id_update)
updated_data = pd.DataFrame([row], columns=columns)
updated_data

,flight_id,origin,destination,departure_date,days_until_departure,base_fare,seats_remaining,capacity,route_popularity,international
0,PA1169,Toronto,Winnipeg,02-11-2027,102,167.0,65,101,0.64,0


# Step 5: Delete Data
*We are not interested in the flight with the lowest popularity, and want to discontinue that route.*

In [98]:
lowest_route_pop = min(flight.route_popularity for flight in all_flights)
lowest_pop_flights = []
for flight in all_flights:
  if flight.route_popularity == lowest_route_pop:
    lowest_pop_flights.append(flight.to_dict())
    
lowest_pop_df = pd.DataFrame(lowest_pop_flights)
lowest_pop_df

,flight_id,origin,destination,departure_date,days_until_departure,base_fare,seats_remaining,capacity,route_popularity,international
0,PA1150,Quebec City,Vancouver,03-06-2027,125,513,155,170,0.5,0
1,PA1151,Quebec City,Vancouver,01-09-2027,69,513,144,190,0.5,0
2,PA1200,Winnipeg,Halifax,03-18-2027,137,256,85,87,0.5,0
3,PA1201,Winnipeg,Halifax,11-22-2026,21,256,52,102,0.5,0
4,PA2696,Ottawa,Seoul,03-13-2027,132,1355,230,297,0.5,1
5,PA2697,Ottawa,Seoul,12-01-2026,30,1355,102,325,0.5,1
6,PA2698,Seoul,Ottawa,02-19-2027,110,1355,244,293,0.5,1
7,PA2699,Seoul,Ottawa,03-07-2027,126,1355,136,328,0.5,1


*Since there are multiple, we want to discontinue the flight that has the largest capacity.*

In [99]:
max_capacity_id = lowest_pop_df.loc[lowest_pop_df["capacity"].idxmax(), "flight_id"]
max_capacity_id

'PA2699'

In [100]:
def delete_flight(flight_id):
    """Delete one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            DELETE FROM flights
            WHERE flight_id = ?
        """, (flight_id,))

In [101]:
delete_flight(max_capacity_id)
row, columns = get_flight(max_capacity_id)
if row is None: 
  print(f"Flight {max_capacity_id} was successfully deleted.")
else: 
  retrieved_data = pd.DataFrame([row], columns)
  retrieved_data

Flight not found.
Flight PA2699 was successfully deleted.
